# M1: Minimal Qwen3

Generate text by hand, without `model.generate()`.

```text
text → tokenizer → Qwen3 → logits → argmax → next token
```

## Setup

Load the tokenizer and the model once. Every cell below reuses them.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

## Tokenizer

Converts text to integer IDs and back. The model only sees IDs.

- **Vocabulary**: a fixed table of token strings and their IDs (about 150k for Qwen3). Its size is the last dimension of the logits.
- **Byte-level BPE**: common words become one token, rare words split into pieces. Any text can be encoded.
- **Special tokens**: control signals such as `<|endoftext|>` and `<|im_end|>`. EOS stops generation.
- Not a neural network. No weights, no GPU.
- Paired with its model: Qwen3 IDs only work with Qwen3.

In [2]:
# encode() returns a list by default; return_tensors="pt" returns a PyTorch tensor
ids = tokenizer.encode("The capital of France is")
print(ids)

# Shape of input_ids: [batch, seq_len]
# torch.Size([1, 5]) means a batch of 1 prompt with 5 tokens
input_ids = tokenizer.encode("The capital of France is", return_tensors="pt")
print(input_ids.shape)

[785, 6722, 315, 9625, 374]
torch.Size([1, 5])


In [3]:
# See how text is split. 'Ġ' marks a leading space.
def show_tokens(text):
    ids = tokenizer.encode(text)
    print(repr(text), "→", tokenizer.convert_ids_to_tokens(ids))


show_tokens("Hello, World")
show_tokens("Hello, Worl")

'Hello, World' → ['Hello', ',', 'ĠWorld']
'Hello, Worl' → ['Hello', ',', 'ĠWor', 'l']


`"Hello, Worl"` ends with `ĠWor` + `l`, a split the model rarely saw in training (it saw `ĠWorld`). So the model does not predict `d` next. The model reads tokens, not text.

In [4]:
# Vocabulary size, seen from three places
print(tokenizer.vocab_size)      # base vocabulary
print(len(tokenizer))            # base + special tokens
print(model.config.vocab_size)   # rows in the embedding table = last dim of logits

151643
151669
151936


In [5]:
# The first IDs in the vocabulary are single bytes
for token_id in range(100):
    print(f"token id {token_id} is {tokenizer.decode(token_id)!r}")

token id 0 is '!'
token id 1 is '"'
token id 2 is '#'
token id 3 is '$'
token id 4 is '%'
token id 5 is '&'
token id 6 is "'"
token id 7 is '('
token id 8 is ')'
token id 9 is '*'
token id 10 is '+'
token id 11 is ','
token id 12 is '-'
token id 13 is '.'
token id 14 is '/'
token id 15 is '0'
token id 16 is '1'
token id 17 is '2'
token id 18 is '3'
token id 19 is '4'
token id 20 is '5'
token id 21 is '6'
token id 22 is '7'
token id 23 is '8'
token id 24 is '9'
token id 25 is ':'
token id 26 is ';'
token id 27 is '<'
token id 28 is '='
token id 29 is '>'
token id 30 is '?'
token id 31 is '@'
token id 32 is 'A'
token id 33 is 'B'
token id 34 is 'C'
token id 35 is 'D'
token id 36 is 'E'
token id 37 is 'F'
token id 38 is 'G'
token id 39 is 'H'
token id 40 is 'I'
token id 41 is 'J'
token id 42 is 'K'
token id 43 is 'L'
token id 44 is 'M'
token id 45 is 'N'
token id 46 is 'O'
token id 47 is 'P'
token id 48 is 'Q'
token id 49 is 'R'
token id 50 is 'S'
token id 51 is 'T'
token id 52 is 'U'
tok

In [6]:
# Qwen3 adds no BOS or EOS when encoding; EOS only appears when the model generates it
eos_token_ids = model.generation_config.eos_token_id  # int or list of ints
if isinstance(eos_token_ids, int):
    eos_token_ids = [eos_token_ids]
print({i: tokenizer.decode(i) for i in eos_token_ids})

{151645: '<|im_end|>', 151643: '<|endoftext|>'}


## Forward Pass

One forward pass turns `input_ids` into logits: a score for every vocabulary token, at every position.

In [7]:
# logits shape: [batch, seq_len, vocab_size]
# torch.Size([1, 5, 151936]) means:
#   1      = 1 prompt
#   5      = 5 tokens in "The capital of France is"
#   151936 = one score for every token in the vocabulary
# Every position predicts the next token. Generation uses only the last one.
input_ids = tokenizer.encode("The capital of France is", return_tensors="pt")
outputs = model(input_ids, use_cache=False)
print(outputs.logits.shape)

torch.Size([1, 5, 151936])


In [8]:
# Inspect the top-k next-token probabilities
last_logits = outputs.logits[:, -1, :]  # all batches, last position, all vocabulary scores
probabilities = torch.softmax(last_logits.float(), dim=-1)  # softmax turns raw scores into probabilities between 0 and 1
top_k = 10
top_probs, top_indices = torch.topk(probabilities, top_k)
print(f"Top {top_k} tokens by probability:")
for i in range(top_k):
    token_id = top_indices[0][i].item()  # token ID ranked i
    probability = top_probs[0][i].item()  # its probability
    token_str = tokenizer.decode(token_id)  # decode the token ID into text
    print(f"Token ID: {token_id}, Token: {token_str!r}, Probability: {probability:.4f}")

Top 10 tokens by probability:
Token ID: 12095, Token: ' Paris', Probability: 0.6687
Token ID: 7407, Token: ' located', Probability: 0.0276
Token ID: 279, Token: ' the', Probability: 0.0215
Token ID: 1112, Token: '...', Probability: 0.0139
Token ID: 220, Token: ' ', Probability: 0.0130
Token ID: 30743, Token: ' ____', Probability: 0.0130
Token ID: 32671, Token: ' ______', Probability: 0.0122
Token ID: 304, Token: ' in', Probability: 0.0108
Token ID: 24209, Token: ' Vers', Probability: 0.0095
Token ID: 2130, Token: '____', Probability: 0.0084


## Greedy Loop

Pick the highest-scoring token, append it, repeat.

- `argmax` works on logits directly. Softmax keeps the order, so the top token is the same.
- Every step recomputes the full sequence, so each step is slower than the last.
- Greedy decoding often falls into a loop and repeats itself.

In [9]:
max_new_tokens = 30

input_ids = tokenizer.encode("The capital of France is", return_tensors="pt")
prompt_len = input_ids.shape[1]

for step in range(max_new_tokens):
    outputs = model(input_ids, use_cache=False)  # recompute the full sequence every step
    last_logits = outputs.logits[:, -1, :]
    next_token = last_logits.argmax(dim=-1, keepdim=True)  # greedy: take the highest score, shape [1, 1]

    token_id = next_token.item()
    print(f"Step {step}: Token ID: {token_id}, Token: {tokenizer.decode(token_id)!r}")

    input_ids = torch.cat([input_ids, next_token], dim=-1)  # append the new token to the input
    if token_id in eos_token_ids:
        break

greedy_output_ids = input_ids[0, prompt_len:]
print(tokenizer.decode(greedy_output_ids))

Step 0: Token ID: 12095, Token: ' Paris'
Step 1: Token ID: 13, Token: '.'
Step 2: Token ID: 576, Token: ' The'
Step 3: Token ID: 6722, Token: ' capital'
Step 4: Token ID: 315, Token: ' of'
Step 5: Token ID: 9625, Token: ' France'
Step 6: Token ID: 374, Token: ' is'
Step 7: Token ID: 1083, Token: ' also'
Step 8: Token ID: 279, Token: ' the'
Step 9: Token ID: 6722, Token: ' capital'
Step 10: Token ID: 315, Token: ' of'
Step 11: Token ID: 279, Token: ' the'
Step 12: Token ID: 5429, Token: ' Republic'
Step 13: Token ID: 315, Token: ' of'
Step 14: Token ID: 9625, Token: ' France'
Step 15: Token ID: 13, Token: '.'
Step 16: Token ID: 576, Token: ' The'
Step 17: Token ID: 6722, Token: ' capital'
Step 18: Token ID: 315, Token: ' of'
Step 19: Token ID: 9625, Token: ' France'
Step 20: Token ID: 374, Token: ' is'
Step 21: Token ID: 1083, Token: ' also'
Step 22: Token ID: 279, Token: ' the'
Step 23: Token ID: 6722, Token: ' capital'
Step 24: Token ID: 315, Token: ' of'
Step 25: Token ID: 279, Token

## Sampling Loop

Same loop, but pick the next token at random, weighted by its probability.

- `temperature` scales the logits before softmax. Below 1 makes the distribution sharper, above 1 makes it flatter.
- `torch.multinomial` draws one token from the probabilities.
- A fixed seed makes the random draws repeatable.

In [10]:
max_new_tokens = 30
temperature = 0.7
torch.manual_seed(0)

input_ids = tokenizer.encode("The capital of France is", return_tensors="pt")
prompt_len = input_ids.shape[1]

for step in range(max_new_tokens):
    outputs = model(input_ids, use_cache=False)
    last_logits = outputs.logits[:, -1, :]
    probabilities = torch.softmax(last_logits.float() / temperature, dim=-1)
    next_token = torch.multinomial(probabilities, num_samples=1)  # random draw, shape [1, 1]

    token_id = next_token.item()
    print(f"Step {step}: Token ID: {token_id}, Token: {tokenizer.decode(token_id)!r}")

    input_ids = torch.cat([input_ids, next_token], dim=-1)
    if token_id in eos_token_ids:
        break

print(tokenizer.decode(input_ids[0, prompt_len:]))

Step 0: Token ID: 12095, Token: ' Paris'
Step 1: Token ID: 13, Token: '.'
Step 2: Token ID: 6771, Token: ' Let'
Step 3: Token ID: 594, Token: "'s"
Step 4: Token ID: 1977, Token: ' say'
Step 5: Token ID: 429, Token: ' that'
Step 6: Token ID: 1052, Token: ' there'
Step 7: Token ID: 374, Token: ' is'
Step 8: Token ID: 264, Token: ' a'
Step 9: Token ID: 1459, Token: ' point'
Step 10: Token ID: 389, Token: ' on'
Step 11: Token ID: 279, Token: ' the'
Step 12: Token ID: 7329, Token: ' surface'
Step 13: Token ID: 315, Token: ' of'
Step 14: Token ID: 279, Token: ' the'
Step 15: Token ID: 9237, Token: ' Earth'
Step 16: Token ID: 594, Token: "'s"
Step 17: Token ID: 7329, Token: ' surface'
Step 18: Token ID: 11, Token: ','
Step 19: Token ID: 323, Token: ' and'
Step 20: Token ID: 429, Token: ' that'
Step 21: Token ID: 582, Token: ' we'
Step 22: Token ID: 614, Token: ' have'
Step 23: Token ID: 311, Token: ' to'
Step 24: Token ID: 9245, Token: ' construct'
Step 25: Token ID: 264, Token: ' a'
Step 26:

## Compare with `model.generate()`

`model.generate()` runs the same loop for us. With `do_sample=False` it is greedy, so it must match the greedy loop above token for token.

In [11]:
input_ids = tokenizer.encode("The capital of France is", return_tensors="pt")
prompt_len = input_ids.shape[1]

generated = model.generate(input_ids, max_new_tokens=30, do_sample=False)  # returns prompt + new tokens

hf_output_ids = generated[0, prompt_len:]
print(tokenizer.decode(hf_output_ids))
print("matches greedy loop:", torch.equal(hf_output_ids, greedy_output_ids))

 Paris. The capital of France is also the capital of the Republic of France. The capital of France is also the capital of the European Union. The
matches greedy loop: True


## Open Questions

- How does the time per step change as the sequence grows? Why?
- Every forward pass computes logits for all positions, but generation uses only the last one. What is wasted?